In [1]:
from collections import Counter, defaultdict
from itertools import product
import math

class SimpleHMMTagger:
    def __init__(self):
        self.pos_tags = ["NOUN", "VERB", "DET", "ADJ", "ADP"]
        self.init_probs = {}
        self.trans_probs = defaultdict(dict)
        self.emit_probs = defaultdict(dict)
        self.vocabulary = set()

    def fit(self, corpus):
        start_counts = Counter()
        tag_counts = Counter()
        trans_counts = defaultdict(Counter)
        emit_counts = defaultdict(Counter)

        for sentence in corpus:
            previous_tag = None
            for idx, (word, tag) in enumerate(sentence):
                self.vocabulary.add(word)
                tag_counts[tag] += 1
                emit_counts[tag][word] += 1

                if idx == 0:
                    start_counts[tag] += 1
                if previous_tag:
                    trans_counts[previous_tag][tag] += 1

                previous_tag = tag

        num_tags = len(self.pos_tags)
        total_sentences = len(corpus)
        vocab_size = len(self.vocabulary)

        self.init_probs = {
            tag: (start_counts[tag] + 1) / (total_sentences + num_tags)
            for tag in self.pos_tags
        }

        for prev in self.pos_tags:
            total = sum(trans_counts[prev].values()) + num_tags
            self.trans_probs[prev] = {
                curr: (trans_counts[prev][curr] + 1) / total
                for curr in self.pos_tags
            }

        for tag in self.pos_tags:
            total = tag_counts[tag] + vocab_size
            self.emit_probs[tag] = {
                word: (emit_counts[tag][word] + 1) / total
                for word in self.vocabulary
            }

    def decode_exhaustive(self, tokens):
        best_tags = None
        max_log_prob = -float("inf")

        for possible_tags in product(self.pos_tags, repeat=len(tokens)):
            log_prob = 0.0

            first_tag = possible_tags[0]
            log_prob += math.log(self.init_probs[first_tag])
            log_prob += math.log(self.emit_probs[first_tag].get(tokens[0], 1e-6))


            for i in range(1, len(tokens)):
                prev_t, curr_t = possible_tags[i-1], possible_tags[i]
                log_prob += math.log(self.trans_probs[prev_t].get(curr_t, 1e-6))
                log_prob += math.log(self.emit_probs[curr_t].get(tokens[i], 1e-6))

            if log_prob > max_log_prob:
                max_log_prob = log_prob
                best_tags = possible_tags

        return list(best_tags)


In [2]:
train_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
    [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
    [("Dogs", "NOUN"), ("bark", "VERB")],
    [("Cats", "NOUN"), ("meow", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"), ("house", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN"), ("sleeps", "VERB")],
]


tagger = SimpleHMMTagger()
tagger.fit(train_data)

test_data = [
    ["The", "dog", "barks"],
    ["A", "cat", "sleeps"],
    ["The", "big", "dog", "runs"],
    ["Dogs", "bark"],
]

for sent in test_data:
    print(f"{sent} → {tagger.decode_exhaustive(sent)}")

['The', 'dog', 'barks'] → ['DET', 'NOUN', 'VERB']
['A', 'cat', 'sleeps'] → ['DET', 'NOUN', 'VERB']
['The', 'big', 'dog', 'runs'] → ['DET', 'ADJ', 'NOUN', 'VERB']
['Dogs', 'bark'] → ['NOUN', 'VERB']
